# معمل تجارب تفكيك الاستعلامات المركبة (Query Decomposition Lab)

أهلا بكم في هذا اللاب العملي الممتع للمحاضرة 51.
في هذا المعمل، سنتعلم كيف نحل واحدة من أصعب مشاكل الـ RAG: **الأسئلة المركبة والمعقدة (Complex Multi-Part Questions)**.

المشكلة ببساطة:
لما المستخدم يسأل سؤال مركب فيه مقارنة مثل:
'كيف تدير LangChain الذاكرة والوكلاء مقارنة بـ CrewAI؟'
لو أرسلنا هذا السؤال ككتلة واحدة لقاعدة المتجهات، النموذج هيتشتت وهيجيب مقاطع ناقصة.

الحل السحري (Query Decomposition):
1. نفكك السؤال المركب إلى 2 أو 4 أسئلة ذرية بسيطة.
2. نسترجع السياق الخاص بكل سؤال فرعي لوحده عبر مسترجع MMR.
3. نولد إجابة مبدئية لكل سؤال فرعي.
4. نجمع الإجابات كلها في مركب إجابات (Synthesizer) يعطينا مقارنة شاملة ومتكاملة.

### 1. تجهيز المكتبات
يلا نستورد المكتبات الأساسية لتقطيع المستندات، بناء الفهرس، والتعامل مع النماذج اللغوية:

In [1]:
import os
import re
from dotenv import load_dotenv

from langchain_core.documents import Document
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings

load_dotenv()
print("تم استيراد كافة المكتبات بنجاح.")

C:\Users\aliashraf\AppData\Local\Temp\ipykernel_18340\4067636141.py:8: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


تم استيراد كافة المكتبات بنجاح.


### 2. تحميل وتقطيع ملف البيانات (LangChain & CrewAI)
هنستخدم نفس ملف البيانات الغني بالمعلومات التقنية عن الذاكرة والوكلاء في اللغتين:

In [2]:
dataset_file = "langchain_crewai_dataset.txt"

loader = TextLoader(dataset_file, encoding="utf-8")
raw_docs = loader.load()

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=400,
    chunk_overlap=50,
    separators=["\n\n", "\n", ". ", " "]
)
chunks = text_splitter.split_documents(raw_docs)

print(f"تم تحميل الملف وتقطيعه إلى {len(chunks)} مقطع بنجاح.")

تم تحميل الملف وتقطيعه إلى 23 مقطع بنجاح.


### 3. بناء الفهرس ومسترجع MMR المتنوع
هننشئ فهرس FAISS مع نموذج التضمين المحلي ونجهز مسترجع MMR لجلب أفضل 3 مقاطع متنوعة لكل سؤال فرعي:

In [3]:
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vector_store = FAISS.from_documents(chunks, embeddings)

retriever = vector_store.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 3, "fetch_k": 10, "lambda_mult": 0.5}
)
print("تم بناء الفهرس وإعداد مسترجع MMR بنجاح.")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

تم بناء الفهرس وإعداد مسترجع MMR بنجاح.


### 4. بناء سلسلة تفكيك السؤال (Decomposition Chain)
هنكتب قالب توجيه يطلب من النموذج اللغوي تحليل السؤال المعقد وتقسيمه إلى أسئلة فرعية مبسطة:

In [4]:
decomp_prompt = PromptTemplate.from_template(
    "You are an AI assistant. Decompose the following complex question into 2 to 4 smaller "
    "sub questions for better document retrieval.\n"
    "Question: {question}\n"
    "Sub questions:"
)

def fallback_decomposer(inputs):
    return (
        "1. What memory mechanisms does LangChain provide?\n"
        "2. How do LangChain agents and tool calling operate?\n"
        "3. What memory architecture does CrewAI support?\n"
        "4. How do CrewAI autonomous agents collaborate?"
    )

groq_key = os.getenv("GROQ_API_KEY")
if groq_key:
    try:
        from langchain_groq import ChatGroq
        decomposer_llm = ChatGroq(model="qwen/qwen3.8-27b", temperature=0.0)
        decomp_chain = decomp_prompt | decomposer_llm | StrOutputParser()
        print("تم ربط سلسلة التفكيك بنموذج ChatGroq بنجاح.")
    except Exception:
        decomp_chain = fallback_decomposer
else:
    decomp_chain = fallback_decomposer

تم ربط سلسلة التفكيك بنموذج ChatGroq بنجاح.


### 5. تجربة تفكيك السؤال وتنظيف المخرجات
تعال نجرب السؤال المركب ونشوف الأسئلة الفرعية الناتجة بعد تنظيف الأرقام والعلامات:

In [5]:
complex_query = "How does LangChain use memory and agents compared to CrewAI?"

try:
    raw_sub_q = decomp_chain.invoke({"question": complex_query})
except Exception:
    raw_sub_q = fallback_decomposer({})

# تنظيف الترقيم والنقاط الزائدة
sub_questions = []
for line in raw_sub_q.strip().split("\n"):
    cleaned = re.sub(r"^\s*(\d+[\.\)]|-|\*)\s*", "", line).strip()
    if cleaned and len(cleaned) > 10:
        sub_questions.append(cleaned)

print("السؤال المركب الأصلي:")
print(complex_query)
print("\nالأسئلة الفرعية الناتجة:")
for i, sq in enumerate(sub_questions, 1):
    print(f"[{i}] {sq}")

السؤال المركب الأصلي:
How does LangChain use memory and agents compared to CrewAI?

الأسئلة الفرعية الناتجة:
[1] How does LangChain implement memory mechanisms and agent architectures?
[2] How does CrewAI structure its memory systems and agent workflows?
[3] What are the key architectural differences between LangChain agents and CrewAI agents?
[4] How do the memory management strategies in LangChain and CrewAI compare in terms of persistence and context handling?


### 6. استرجاع المقاطع والإجابة عن كل سؤال فرعي
الآن سنمر على كل سؤال فرعي، نسترجع مقاطعه الخاصة عبر مسترجع MMR، ونطلب من النموذج توليد إجابة مخصصة:

In [6]:
try:
    from langchain.chains.combine_documents import create_stuff_documents_chain
except (ImportError, ModuleNotFoundError):
    from langchain_classic.chains.combine_documents import create_stuff_documents_chain

qa_prompt = ChatPromptTemplate.from_template(
    "Use the context below to answer the question concisely.\n"
    "Context:\n{context}\n\n"
    "Question: {input}\n"
    "Answer:"
)

def fallback_sub_answer(inputs):
    q = inputs["input"].lower()
    if "langchain" in q and "memory" in q:
        return "LangChain provides ConversationBufferMemory and ConversationSummaryMemory for chat persistence."
    elif "langchain" in q and "agent" in q:
        return "LangChain agents use StructuredTool and ToolNode interfaces to call external APIs."
    elif "crewai" in q and "memory" in q:
        return "CrewAI supports a multi-tiered memory system with Short-Term, Long-Term, and Entity Memory."
    elif "crewai" in q and "agent" in q:
        return "CrewAI agents are role-playing entities configured with role, goal, backstory, and tools."
    return "Grounded response based on retrieved documentation."

if groq_key:
    try:
        qa_llm = ChatGroq(model="qwen/qwen3.8-27b", temperature=0.0)
        qa_chain = create_stuff_documents_chain(qa_llm, qa_prompt)
    except Exception:
        qa_chain = fallback_sub_answer
else:
    qa_chain = fallback_sub_answer

sub_answers = []
for sq in sub_questions:
    docs = retriever.invoke(sq)
    try:
        ans = qa_chain.invoke({"input": sq, "context": docs})
    except Exception:
        ans = fallback_sub_answer({"input": sq, "context": docs})
    sub_answers.append((sq, str(ans)))

for idx, (sq, ans) in enumerate(sub_answers, 1):
    print(f"--- السؤال الفرعي [{idx}] ---")
    print(f"س: {sq}")
    print(f"ج: {ans}\n")

--- السؤال الفرعي [1] ---
س: How does LangChain implement memory mechanisms and agent architectures?
ج: LangChain implements memory mechanisms using built-in classes like `ConversationBufferMemory` to persist raw chat history across turns, and it structures agents as cognitive architectures where language models dynamically decide on tool execution by leveraging `StructuredTool` interfaces that bind Python functions with strict Pydantic schemas.

--- السؤال الفرعي [2] ---
س: How does CrewAI structure its memory systems and agent workflows?
ج: CrewAI structures its memory system using a multi-tiered architecture consisting of Short-Term, Long-Term, and Entity Memory. For agent workflows, it utilizes the Crew primitive to coordinate agents and tasks, supports asynchronous execution for concurrent work, and offers a Hierarchical Process where an automated Manager Agent delegates sub-tasks and reviews outputs against requirements.

--- السؤال الفرعي [3] ---
س: What are the key architectura

### 7. دمج الإجابات الفرعية في إجابة موحدة (Synthesis Layer)
في المرحلة الأخيرة، نمرر كل الأسئلة والإجابات الفرعية لنموذج التوليف ليصيغ مقارنة متكاملة تجيب عن السؤال الأصلي بأعلى درجات التنسيق:

In [7]:
synthesis_prompt = PromptTemplate.from_template(
    "You are an expert AI architect. Synthesize the following question-answer pairs into a "
    "coherent, cohesive final comparative answer for the overarching question.\n\n"
    "Overarching Question: {original_query}\n\n"
    "Sub-Questions and Verified Answers:\n{sub_answers}\n\n"
    "Synthesized Comprehensive Answer:"
)

formatted_pairs = "\n\n".join([f"Q: {q}\nA: {a}" for q, a in sub_answers])

def fallback_synthesizer():
    return (
        "Both LangChain and CrewAI possess robust memory and agent functionality. "
        "LangChain excels at granular, component-level orchestration with memory modules "
        "like ConversationBufferMemory and StructuredTool agents. Conversely, CrewAI specializes "
        "in multi-agent collaboration with a multi-tiered memory architecture (Short-Term, "
        "Long-Term, Entity) and role-playing autonomous agents executing collaborative workflows."
    )

if groq_key:
    try:
        synth_llm = ChatGroq(model="qwen/qwen3.8-27b", temperature=0.0)
        synth_chain = synthesis_prompt | synth_llm | StrOutputParser()
        final_answer = synth_chain.invoke({
            "original_query": complex_query,
            "sub_answers": formatted_pairs
        })
    except Exception:
        final_answer = fallback_synthesizer()
else:
    final_answer = fallback_synthesizer()

print("=" * 70)
print("            الإجابة النهائية المركبة والمقارنة الشاملة")
print("=" * 70)
print(final_answer)
print("=" * 70)

            الإجابة النهائية المركبة والمقارنة الشاملة
LangChain and CrewAI approach memory and agent orchestration with fundamentally different architectural philosophies, reflecting their distinct design goals: LangChain prioritizes low-level flexibility and granular control, while CrewAI focuses on high-level, role-based multi-agent collaboration.

**Memory Mechanisms**
The two frameworks diverge significantly in how they manage context and persistence:
*   **LangChain** integrates memory modules directly into chains and state channels. It offers flexible options ranging from `ConversationBufferMemory` for persisting raw chat history to `VectorStoreRetrieverMemory`, which uses semantic similarity searches to retrieve only dialogue fragments relevant to the immediate query. This approach allows for precise, developer-controlled context handling.
*   **CrewAI** employs a structured, multi-tiered persistence architecture consisting of **Short-Term**, **Long-Term**, and **Entity Memory*

### 8. جرب بنفسك!
الآن حان دورك لتجربة سؤال مركب جديد:
1. جرب سؤالاً مثل:
   `"What are the execution processes in CrewAI and how do they differ from LangChain LCEL chains?"`
2. لاحظ كيف سيقوم النظام بتفكيكه إلى:
   - سؤال عن عمليات CrewAI (Sequential & Hierarchical).
   - سؤال عن سلاسل LangChain LCEL.
   - سؤال يقارن بين الفلسفتين!

In [8]:
# اكتب سؤالك المركب هنا
my_complex_query = "What are the execution processes in CrewAI and how do they differ from LangChain LCEL chains?"

print("جاري تفكيك سؤالك التجريبي...")
try:
    my_raw_sub = decomp_chain.invoke({"question": my_complex_query})
    my_sub_questions = [
        re.sub(r"^\s*(\d+[\.\)]|-|\*)\s*", "", line).strip()
        for line in my_raw_sub.strip().split("\n")
        if len(line.strip()) > 10
    ]
except Exception:
    my_sub_questions = [
        "What are the execution processes in CrewAI?",
        "How do LangChain LCEL chains execute workflows?",
        "What is the difference between CrewAI processes and LangChain LCEL?"
    ]

print(f"تم تفكيك السؤال إلى {len(my_sub_questions)} أسئلة فرعية:")
for i, sq in enumerate(my_sub_questions, 1):
    print(f"  {i}. {sq}")

جاري تفكيك سؤالك التجريبي...


تم تفكيك السؤال إلى 3 أسئلة فرعية:
  1. What are the execution processes and workflow mechanisms in CrewAI?
  2. How do LangChain LCEL (LangChain Expression Language) chains function and what are their execution characteristics?
  3. What are the key architectural and operational differences between CrewAI's execution model and LangChain LCEL chains?
